We had gotten feedback to try using an elastic net regression model.

In [72]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
DATA_FILE = "../../data/processed/cardio_onc_prostate_06_broad_clean.csv"
OUT_DIR = "Results/ElasticNet"
TARGET = "at_risk"
SEED = 42

os.makedirs(OUT_DIR, exist_ok = True)

In [ ]:
df = pd.read_csv(DATA_FILE)

df["at_risk"] = (
    (df["bp_meds_post_binary"] == 1) |
    (df["lipid_meds_post_binary"] == 1) |
    (df["dm_meds_post_binary"] == 1)
).astype(int)

In [ ]:
continuous_features = [
    "bmi", "age", "sbp", "dbp", "ascvd_10yr"
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features = [
    "ethnicity_enc", "specific_nht_used_enc",
    "adt_agent_enc", "prescribing_provider_enc",
]

all_features = continuous_features + binary_features + encoded_features

In [ ]:
X_df = df[all_features].copy().astype(float)
y = df[TARGET].values.astype(int)
X_arr = X_df.values

## Preprocess

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cont", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), continuous_features),

        ("binary", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
        ]), binary_features),

        ("encoded", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
        ]), encoded_features),
    ],
    remainder="drop",
)

X_processed = preprocessor.fit_transform(X_df)

## Model Fit

In [ ]:
model = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.1, 0.5, 0.9],
    Cs=np.logspace(-4, 2, 50),
    cv=5,
    scoring='roc_auc',
    max_iter=10000,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_processed, y)

LogisticRegressionCV(Cs=array([1.00000000e-04, 1.32571137e-04, 1.75751062e-04, 2.32995181e-04,
       3.08884360e-04, 4.09491506e-04, 5.42867544e-04, 7.19685673e-04,
       9.54095476e-04, 1.26485522e-03, 1.67683294e-03, 2.22299648e-03,
       2.94705170e-03, 3.90693994e-03, 5.17947468e-03, 6.86648845e-03,
       9.10298178e-03, 1.20679264e-02, 1.59985872e-02, 2.12095089e-02,
       2.81176870e-02, 3.72...
       8.28642773e-01, 1.09854114e+00, 1.45634848e+00, 1.93069773e+00,
       2.55954792e+00, 3.39322177e+00, 4.49843267e+00, 5.96362332e+00,
       7.90604321e+00, 1.04811313e+01, 1.38949549e+01, 1.84206997e+01,
       2.44205309e+01, 3.23745754e+01, 4.29193426e+01, 5.68986603e+01,
       7.54312006e+01, 1.00000000e+02]),
                     class_weight='balanced', cv=5, l1_ratios=[0.1, 0.5, 0.9],
                     max_iter=10000, penalty='elasticnet', random_state=42,
                     scoring='roc_auc', solver='saga')

## Extract Associations

In [79]:
# Get feature names from the preprocessor
onehot_features = preprocessor.named_transformers_['encoded']['onehot'].get_feature_names_out(encoded_features).tolist()
all_processed_features = continuous_features + binary_features + onehot_features

# Build the dataframe
coef_df = pd.DataFrame({
    'feature':    all_processed_features,
    'log_odds':   model.coef_[0],
    'odds_ratio': np.exp(model.coef_[0])
}).query('log_odds != 0').sort_values('log_odds', key=abs, ascending=False)

## Errors & P-values

In [80]:
X_processed_df = pd.DataFrame(X_processed, columns=all_processed_features)

selected = coef_df['feature'].tolist()
X_sel = sm.add_constant(X_processed_df[selected])

logit_model = sm.Logit(y, X_sel).fit()
print(logit_model.summary())

odds_ratios = pd.DataFrame({
    'OR':     np.exp(logit_model.params),
    'CI_low': np.exp(logit_model.conf_int()[0]),
    'CI_high':np.exp(logit_model.conf_int()[1]),
    'p_value':logit_model.pvalues
})

Optimization terminated successfully.
         Current function value: 0.549146
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  239
Model:                          Logit   Df Residuals:                      233
Method:                           MLE   Df Model:                            5
Date:                Mon, 18 May 2026   Pseudo R-squ.:                  0.1346
Time:                        07:44:32   Log-Likelihood:                -131.25
converged:                       True   LL-Null:                       -151.66
Covariance Type:            nonrobust   LLR p-value:                 1.016e-07
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -1.3700      0.323     -4.240      0.000      -2.003      -0.737
dm

## Conclusion

Elastic net zeroed out all features and only kept the intercept. Model found no feature informative enough to keep.